# 🧠 Mind Mate — Emotion Classifier Fine-Tuning

**FYP: Mind Mate — Student Wellness Web App**

This notebook fine-tunes `distilroberta-base` on three datasets to classify student text into **5 mood levels** used by Mind Mate:

| Label | Mood | Score |
|-------|------|-------|
| 0 | awful | 1/10 |
| 1 | bad | 3/10 |
| 2 | okay | 5/10 |
| 3 | good | 8/10 |
| 4 | great | 10/10 |

**Datasets used:**
- GoEmotions (43k Google Reddit comments, 27 emotions)
- ISEAR (7k cross-cultural emotion survey, 4 emotions)
- Mental Health Reddit (94k posts, 7 mental health categories)

**Expected training time:** ~25 minutes on Colab T4 GPU

---
> ⚠️ Make sure you: **Runtime → Change runtime type → T4 GPU** before running

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install transformers datasets torch scikit-learn pandas pyarrow huggingface_hub accelerate evaluate seqeval -q

## Step 2 — Login to Hugging Face Hub

You need a free HF account. Get your token from: https://huggingface.co/settings/tokens (use **write** access)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 3 — Upload Your Dataset Files

Run this cell, then upload these 3 files from your PC:
- `train-00000-of-00001.parquet` (GoEmotions train)
- `eng_dataset.csv` (ISEAR)
- `Combined Data.csv` (Mental Health Reddit)

In [ ]:
from google.colab import files
import io

print('Upload your 3 dataset files now:')
print('  1. train-00000-of-00001.parquet  (GoEmotions)')
print('  2. eng_dataset.csv               (ISEAR)')
print('  3. Combined Data.csv             (Mental Health Reddit)')
print()
uploaded = files.upload()
print('\nUploaded files:', list(uploaded.keys()))

## Step 4 — Load and Preview Datasets

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import io

# ── GoEmotions ──
go_df = pq.read_table(io.BytesIO(uploaded['train-00000-of-00001.parquet'])).to_pandas()
print(f'GoEmotions: {len(go_df):,} rows')
print(go_df.head(3)[['text','labels']])
print()

# ── ISEAR ──
isear_df = pd.read_csv(io.BytesIO(uploaded['eng_dataset.csv']), encoding='utf-8-sig')
print(f'ISEAR: {len(isear_df):,} rows')
print(isear_df['sentiment'].value_counts())
print()

# ── Mental Health Reddit ──
mh_df = pd.read_csv(io.BytesIO(uploaded['Combined Data.csv']))
print(f'Mental Health Reddit: {len(mh_df):,} rows')
print(mh_df['status'].value_counts())

## Step 5 — Map All Labels to 5 Mind Mate Moods

In [ ]:
import numpy as np

# Mind Mate mood labels
MOOD_LABELS  = ['awful', 'bad', 'okay', 'good', 'great']
MOOD_TO_ID   = {m: i for i, m in enumerate(MOOD_LABELS)}

# ── GoEmotions: 28 emotions → 5 moods ──
# 28 GoEmotions emotion names (index 0-27)
GO_EMOTIONS = [
    'admiration','amusement','anger','annoyance','approval',
    'caring','confusion','curiosity','desire','disappointment',
    'disapproval','disgust','embarrassment','excitement','fear',
    'gratitude','grief','joy','love','nervousness',
    'neutral','optimism','pride','realization','relief',
    'remorse','sadness','surprise'
]

# Map each emotion index to a Mind Mate mood
GO_MOOD_MAP = {
    0:  'great',  # admiration
    1:  'great',  # amusement
    2:  'awful',  # anger
    3:  'awful',  # annoyance
    4:  'good',   # approval
    5:  'good',   # caring
    6:  'okay',   # confusion
    7:  'okay',   # curiosity
    8:  'good',   # desire
    9:  'bad',    # disappointment
    10: 'bad',    # disapproval
    11: 'awful',  # disgust
    12: 'bad',    # embarrassment
    13: 'great',  # excitement
    14: 'bad',    # fear
    15: 'great',  # gratitude
    16: 'bad',    # grief
    17: 'great',  # joy
    18: 'great',  # love
    19: 'bad',    # nervousness
    20: 'okay',   # neutral
    21: 'great',  # optimism
    22: 'great',  # pride
    23: 'okay',   # realization
    24: 'great',  # relief
    25: 'bad',    # remorse
    26: 'bad',    # sadness
    27: 'good',   # surprise
}

def go_labels_to_mood(label_list):
    """For multi-label, pick the most extreme mood (awful > great > bad > good > okay)"""
    priority = {'awful': 5, 'great': 4, 'bad': 3, 'good': 2, 'okay': 1}
    moods = [GO_MOOD_MAP[l] for l in label_list if l in GO_MOOD_MAP]
    if not moods:
        return 'okay'
    return max(moods, key=lambda m: priority.get(m, 0))

go_df['mood'] = go_df['labels'].apply(go_labels_to_mood)
go_processed = go_df[['text', 'mood']].copy()
print('GoEmotions mood distribution:')
print(go_processed['mood'].value_counts())
print()

# ── ISEAR: 4 emotions → 5 moods ──
ISEAR_MAP = {'joy': 'great', 'anger': 'awful', 'fear': 'bad', 'sadness': 'bad'}
isear_df = isear_df.dropna(subset=['content', 'sentiment'])
isear_df = isear_df[isear_df['sentiment'].isin(ISEAR_MAP.keys())].copy()
isear_df['mood'] = isear_df['sentiment'].map(ISEAR_MAP)
isear_processed = isear_df[['content', 'mood']].rename(columns={'content': 'text'})
print('ISEAR mood distribution:')
print(isear_processed['mood'].value_counts())
print()

# ── Mental Health Reddit → 5 moods ──
MH_MAP = {
    'Normal':               'okay',
    'Anxiety':              'bad',
    'Stress':               'bad',
    'Depression':           'awful',
    'Suicidal':             'awful',
    'Bipolar':              'bad',
    'Personality disorder': 'bad',
}
mh_df = mh_df.dropna(subset=['statement', 'status'])
mh_df = mh_df[mh_df['status'].isin(MH_MAP.keys())].copy()
mh_df['mood'] = mh_df['status'].map(MH_MAP)
# Subsample to balance — mental health dataset is huge
mh_processed = mh_df[['statement', 'mood']].rename(columns={'statement': 'text'})
# Cap at 15k per class to avoid imbalance
mh_processed = mh_processed.groupby('mood', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 15000), random_state=42)
).reset_index(drop=True)
print('Mental Health Reddit mood distribution (after balancing):')
print(mh_processed['mood'].value_counts())

## Step 6 — Combine, Clean & Balance the Full Dataset

In [ ]:
# Combine all datasets
combined = pd.concat([
    go_processed,
    isear_processed,
    mh_processed
], ignore_index=True)

# Clean text
combined['text'] = combined['text'].astype(str).str.strip()
combined = combined[combined['text'].str.len() > 5]  # remove empty/tiny
combined = combined.dropna(subset=['text', 'mood'])

# Add numeric label
combined['label'] = combined['mood'].map(MOOD_TO_ID)
combined = combined.dropna(subset=['label'])
combined['label'] = combined['label'].astype(int)

print(f'Total samples: {len(combined):,}')
print('\nMood distribution:')
for mood, cnt in combined['mood'].value_counts().items():
    pct = cnt / len(combined) * 100
    print(f'  {mood:6s}: {cnt:6,}  ({pct:.1f}%)')

# Shuffle
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)
combined.head()

## Step 7 — Split into Train / Validation / Test Sets

In [ ]:
from sklearn.model_selection import train_test_split

# 80% train, 10% val, 10% test
train_df, temp_df = train_test_split(
    combined, test_size=0.2, stratify=combined['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

print(f'Train: {len(train_df):,}')
print(f'Val:   {len(val_df):,}')
print(f'Test:  {len(test_df):,}')

## Step 8 — Tokenize with DistilRoBERTa

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'distilroberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

# Convert to HF Dataset format
train_ds = Dataset.from_pandas(train_df[['text','label']]).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df[['text','label']]).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df[['text','label']]).map(tokenize, batched=True)

# Set format for PyTorch
cols = ['input_ids', 'attention_mask', 'label']
train_ds.set_format('torch', columns=cols)
val_ds.set_format('torch', columns=cols)
test_ds.set_format('torch', columns=cols)

print('Tokenization complete!')
print('Sample input_ids length:', len(train_ds[0]['input_ids']))

## Step 9 — Load Model & Define Metrics

In [ ]:
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

NUM_LABELS = 5

id2label = {i: m for i, m in enumerate(MOOD_LABELS)}
label2id = {m: i for i, m in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc  = accuracy_score(labels, preds)
    f1   = f1_score(labels, preds, average='weighted')
    return {'accuracy': round(acc, 4), 'f1_weighted': round(f1, 4)}

print(f'Model loaded: {MODEL_NAME}')
print(f'Labels: {id2label}')

## Step 10 — Training Configuration

In [ ]:
from transformers import TrainingArguments, Trainer

# Your HF username — change this!
HF_USERNAME = 'YOUR_HF_USERNAME'   # ← CHANGE THIS to your HF username
MODEL_REPO  = f'{HF_USERNAME}/mindmate-emotion-classifier'

training_args = TrainingArguments(
    output_dir='./mindmate-emotion-classifier',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    greater_is_better=True,
    logging_steps=100,
    push_to_hub=True,
    hub_model_id=MODEL_REPO,
    report_to='none',      # disable wandb
    fp16=True,             # faster on T4 GPU
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print(f'Will push to: https://huggingface.co/{MODEL_REPO}')
print(f'Training on {len(train_ds):,} examples for 3 epochs')
print(f'Estimated time: ~20-25 min on T4 GPU')

## Step 11 — Train the Model ⏳

> This takes ~20-25 minutes on Colab T4. Don't close the tab!

In [ ]:
print('Starting training...')
trainer.train()
print('\n✅ Training complete!')

## Step 12 — Evaluate on Test Set

In [ ]:
import numpy as np

print('=== Test Set Evaluation ===')
results = trainer.predict(test_ds)
preds  = np.argmax(results.predictions, axis=-1)
labels = results.label_ids

print(classification_report(
    labels, preds,
    target_names=MOOD_LABELS,
    digits=4
))

overall_acc = accuracy_score(labels, preds)
overall_f1  = f1_score(labels, preds, average='weighted')
print(f'Overall Accuracy : {overall_acc:.4f}  ({overall_acc*100:.2f}%)')
print(f'Weighted F1 Score: {overall_f1:.4f}')

## Step 13 — Save Model Card & Push to Hugging Face Hub

In [ ]:
# Create a model card for HF Hub
model_card = f"""---
language: en
license: mit
tags:
  - text-classification
  - emotion-detection
  - student-wellness
  - mental-health
datasets:
  - go_emotions
  - isear
  - mental-health-reddit
metrics:
  - accuracy
  - f1
---

# MindMate Emotion Classifier

Fine-tuned `distilroberta-base` for student mood classification into 5 levels.

Built as part of **Mind Mate** — a student wellness web app (FYP).

## Labels
| ID | Label | Description |
|----|-------|-------------|
| 0 | awful | Very negative / distress |
| 1 | bad   | Negative emotions |
| 2 | okay  | Neutral / mixed |
| 3 | good  | Positive |
| 4 | great | Very positive / joyful |

## Training Data
- **GoEmotions** (Google, 43k samples) — 27 emotion labels mapped to 5 moods
- **ISEAR** (7k samples) — cross-cultural emotion survey
- **Mental Health Reddit** (selected 53k samples) — real student mental health posts

## Performance
- Accuracy: {overall_acc:.4f}
- Weighted F1: {overall_f1:.4f}

## Usage
```python
from transformers import pipeline
classifier = pipeline('text-classification', model='{MODEL_REPO}')
result = classifier('I aced my exam today!')
# [{{'label': 'great', 'score': 0.94}}]
```
"""

with open('./mindmate-emotion-classifier/README.md', 'w') as f:
    f.write(model_card)

# Push everything to hub
trainer.push_to_hub(commit_message='Final model — trained on GoEmotions + ISEAR + MentalHealth Reddit')
tokenizer.push_to_hub(MODEL_REPO)

print(f'\n✅ Model pushed to: https://huggingface.co/{MODEL_REPO}')

## Step 14 — Test Your Model Live

In [ ]:
from transformers import pipeline

classifier = pipeline('text-classification', model=f'./mindmate-emotion-classifier')

test_sentences = [
    'I aced my exam today! So happy!',
    'Had a pretty normal day, nothing special.',
    'I am so stressed about this deadline.',
    'I feel completely hopeless and overwhelmed.',
    'Hanging out with friends was amazing!',
    'My assignment got rejected, feeling down.',
    "Can't sleep, too anxious about tomorrow.",
]

print('=== Live Predictions ===')
for text in test_sentences:
    result = classifier(text)[0]
    score_pct = result['score'] * 100
    mood_scores = {'awful':1,'bad':3,'okay':5,'good':8,'great':10}
    mood_val = mood_scores.get(result['label'], 5)
    print(f'  Input : {text}')
    print(f'  Mood  : {result["label"]:5s} ({mood_val}/10)  — confidence: {score_pct:.1f}%')
    print()

## Step 15 — Update Your Mind Mate API

Once the model is on Hugging Face, update `api/analyzeMood.js` in your project:

Change this line:
```js
// OLD — generic Twitter sentiment model
'https://api-inference.huggingface.co/models/cardiffnlp/twitter-roberta-base-sentiment'
```

To your fine-tuned model:
```js
// NEW — your fine-tuned Mind Mate model
`https://api-inference.huggingface.co/models/${MODEL_REPO}`
```

And update the label parsing (old model used LABEL_0/LABEL_2, yours uses mood names directly):
```js
// The response now returns: [{ label: 'great', score: 0.94 }]
const moodLabel = data[0][0].label;  // 'awful'|'bad'|'okay'|'good'|'great'
const moodScore = data[0][0].score;
```

---
**Your model URL will be:**
```
https://huggingface.co/YOUR_HF_USERNAME/mindmate-emotion-classifier
```

## 📊 FYP Presentation Notes

Key points to mention:

1. **Dataset**: Combined 3 real-world datasets totalling ~100k+ labelled text samples
2. **Model**: Fine-tuned `distilroberta-base` (82M parameters) — compact enough for production inference
3. **Task**: 5-class ordinal emotion classification mapped to Mind Mate's mood scale
4. **Label mapping**: Documented mapping from 27 GoEmotions → 5 moods (academically justifiable)
5. **Evaluation**: Accuracy + Weighted F1 on held-out test set (not seen during training)
6. **Deployment**: Model served via Hugging Face Inference API, called from Vercel serverless function
7. **Architecture**: Client → Firebase Hosting → Vercel API → HF Inference → Firebase RTDB

Expected accuracy range: **75-85%** (distilroberta on mixed emotion data typically achieves this)
